# 가상 참여자 노트북 (교차검증 전에 스케일러를 전체 데이터에 맞춤)

In [1]:
import pandas as pd
import numpy as np
import joblib
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(train.shape, test.shape)

(1279, 13) (320, 12)


In [2]:
def display_scores(scores):
    print('Scores:', scores)
    print('Mean:', scores.mean())
    print('Standard deviation:', scores.std())

In [3]:
before = len(train)
train = train.drop_duplicates(subset=[c for c in train.columns if c not in ('id', 'quality')])
print('같은 와인 행 제거:', before, '->', len(train))
X, y = train.drop(columns=['id', 'quality']), train['quality']
X_test = test.drop(columns=['id'])

같은 와인 행 제거: 1279 -> 1083


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # CV 전에 전체 데이터로 맞춤
cv = KFold(n_splits=10, shuffle=True, random_state=0)
scores = np.sqrt(-cross_val_score(SVR(), X_scaled, y, scoring='neg_mean_squared_error', cv=cv))
display_scores(scores)
model = make_pipeline(StandardScaler(), SVR(C=2.0)).fit(X, y)
pred = model.predict(X_test)

Scores: [0.66557195 0.67321368 0.66126418 0.63131512 0.63599319 0.64774478
 0.70235046 0.69732331 0.65862607 0.62620966]
Mean: 0.6599612399118826
Standard deviation: 0.024671512812381904


In [5]:
pd.DataFrame({'id': test['id'], 'quality': pred}).to_csv('submission.csv', index=False)
joblib.dump(model, 'model.pkl')

['model.pkl']